In [1]:
import numpy as np
import pandas as pd
import random
from datetime import datetime, timedelta
import joblib
from tensorflow.keras.models import Sequential, Model
from tensorflow.keras.layers import LSTM, Dense, Dropout, Input, Bidirectional, GlobalAveragePooling1D, BatchNormalization, Conv1D, MaxPooling1D, Attention, Masking
from tensorflow.keras.optimizers import Adam
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
import warnings
warnings.filterwarnings('ignore')
import os
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf

class EnhancedTargetSimulator:
    def __init__(self, track_class):
        self.track_class = track_class
        self.state = "CRUISING"
        self.track_id = random.randint(1000, 9999)
        self._set_initial_conditions()

    def _set_initial_conditions(self):
        if self.track_class == "CIVILIAN":
            self.pos = np.array([np.random.uniform(-50000, 50000), np.random.uniform(-50000, 50000), np.random.uniform(8000, 15000)])
            self.vel = np.array([np.random.uniform(200, 400), np.random.uniform(-50, 50), np.random.uniform(-10, 10)])
        elif self.track_class == "MISSILE":
            self.pos = np.array([np.random.uniform(-20000, 20000), np.random.uniform(-20000, 20000), np.random.uniform(5000, 25000)])
            self.vel = np.array([np.random.uniform(300, 800), np.random.uniform(-100, 100), np.random.uniform(-200, -50)])
        elif self.track_class == "HELICOPTER":
            self.pos = np.array([np.random.uniform(-30000, 30000), np.random.uniform(-30000, 30000), np.random.uniform(100, 5000)])
            self.vel = np.array([np.random.uniform(40, 120), np.random.uniform(-30, 30), np.random.uniform(-5, 5)])
        elif self.track_class == "FIGHTER_JET":
            self.pos = np.array([np.random.uniform(-50000, 50000), np.random.uniform(-50000, 50000), np.random.uniform(1000, 20000)])
            self.vel = np.array([np.random.uniform(150, 850), np.random.uniform(-100, 100), np.random.uniform(-50, 50)])
        elif self.track_class == "SKIFF":
            self.pos = np.array([np.random.uniform(-50000, 50000), np.random.uniform(-50000, 50000), 0.0])
            self.vel = np.array([np.random.uniform(2, 30), np.random.uniform(-10, 10), 0.0])

    def update_state(self, t):
        if self.track_class == "MISSILE":
            if t < 5: self.state = "LAUNCH"
            elif t > 80: self.state = "TERMINAL_DIVE"
            elif t > 60 and np.random.random() < 0.1: self.state = "TARGET_LOCKED"
            else: self.state = "CRUISING"
        elif self.track_class == "CIVILIAN" and t > 90:
            self.state = "LANDING"
        elif self.track_class == "FIGHTER_JET":
            if random.random() < 0.1:
                states = ["CRUISING", "MANEUVERING", "HIGH_G_MANEUVER", "TARGET_LOCKED"]
                self.state = random.choice(states)
            else: self.state = "CRUISING"
        elif self.track_class == "HELICOPTER" and t > 30 and random.random() < 0.2:
            self.state = "AGGRESSIVE_TURN"
        else:
            if random.random() < 0.05:
                states = ["CRUISING", "MANEUVERING", "SLOW_DOWN"]
                self.state = random.choice(states)
            else: self.state = "CRUISING"

    def get_next_step(self, t, dt):
        self.update_state(t)
        prev_vel = self.vel.copy()

        if self.track_class == "FIGHTER_JET":
            if self.state == "HIGH_G_MANEUVER":
                acceleration_magnitude = random.uniform(80, 120)
                angle_change = np.radians(random.uniform(-60, 60))
                c, s = np.cos(angle_change), np.sin(angle_change)
                vx, vy = self.vel[0], self.vel[1]
                self.vel[0], self.vel[1] = vx*c - vy*s, vx*s + vy*c
                speed_direction = np.arctan2(self.vel[1], self.vel[0])
                self.vel[0] += np.cos(speed_direction) * acceleration_magnitude * dt
                self.vel[1] += np.sin(speed_direction) * acceleration_magnitude * dt
                speed = np.linalg.norm(self.vel)
                if speed > 850:
                    self.vel = self.vel * (850 / speed)
            elif self.state == "TARGET_LOCKED": pass
            else:
                acceleration_noise = np.random.normal(0, 1, 3)
                self.vel += acceleration_noise
        elif self.track_class == "MISSILE":
            if t > 80: self.vel[2] -= 80 * dt
        elif self.state == "TERMINAL_DIVE":
            self.vel[2] -= 80 * dt
            self.vel[0] *= 0.995
            self.vel[1] *= 0.995
        elif self.state == "LANDING":
            self.vel[2] -= 15 * dt
            self.vel[0] *= 0.97
            self.vel[1] *= 0.97
            self.vel[2] = max(0, self.vel[2])
        elif self.state == "HOVER" or self.state == "LOITER":
            noise = np.random.normal(0, 2, 3)
            self.vel += noise
            self.vel = np.clip(self.vel, -50, 50)
        elif self.state == "AGGRESSIVE_TURN":
            angle_change = np.radians(45 + random.uniform(-15, 15))
            c, s = np.cos(angle_change), np.sin(angle_change)
            vx, vy = self.vel[0], self.vel[1]
            self.vel[0], self.vel[1] = vx*c - vy*s, vx*s + vy*c
        elif self.state == "TARGET_LOCKED": pass
        else:
            if self.track_class != "MISSILE":
                acceleration_noise = np.random.normal(0, 2, 3)
                self.vel += acceleration_noise

        if self.track_class == "MISSILE":
            self.vel[0] = max(200, abs(self.vel[0])) * np.sign(self.vel[0])
        elif self.track_class == "SKIFF":
            self.vel[2] = 0

        if self.track_class != "MISSILE" and self.track_class != "FIGHTER_JET":
            self.vel += np.random.normal(0, 1, 3) * (1 + t/50.0)

        self.pos += self.vel * dt

        if self.track_class == "SKIFF":
            self.pos[2] = max(0, self.pos[2]) + np.sin(t * 0.5) * 1.5 + random.uniform(-0.5, 0.5)

        accel = (self.vel - prev_vel) / dt if dt > 0 else np.zeros(3)
        jerk_mag = np.linalg.norm(accel) * random.uniform(0.8, 1.2) if len(accel) > 0 else 0

        return self.pos.copy(), self.vel.copy(), accel.copy(), jerk_mag

def generate_spherical_measurements(pos_x, pos_y, pos_z):
    r_true = np.sqrt(pos_x**2 + pos_y**2 + pos_z**2)
    if r_true == 0: return 0.0, 0.0, 0.0, 0.0
    noise_factor = max(1, r_true / 10000)
    sigma_r = 20.0 * noise_factor
    sigma_theta = 0.4 + (r_true / 50000)
    sigma_phi = 0.8 + (r_true / 100000)
    theta_true = np.arctan2(pos_y, pos_x)
    phi_true = np.arcsin(np.clip(pos_z / r_true, -1, 1))

    r_meas = max(0, r_true + np.random.normal(0, sigma_r))
    theta_meas = theta_true + np.random.normal(0, np.radians(sigma_theta))
    phi_meas = phi_true + np.random.normal(0, np.radians(sigma_phi))
    bearing = (np.degrees(theta_meas) + 360) % 360

    if random.random() < 0.02: return -1, -1, -1, -1
    return r_meas, theta_meas, phi_meas, bearing

def generate_enhanced_dataset(num_tracks=300, duration=150):
    tracks_per_class = { "CIVILIAN": 200, "MISSILE": 200, "HELICOPTER": 200, "FIGHTER_JET": 200, "SKIFF": 200 }
    simulators = []
    for class_name, count in tracks_per_class.items():
        simulators.extend([EnhancedTargetSimulator(class_name) for _ in range(count)])

    base_timestamp = 3991182426
    ntp_times = [base_timestamp + i * 10 for i in range(duration)]
    all_data = []

    for t in range(duration):
        current_ntp_time = ntp_times[t]
        for sim in simulators:
            pos, vel, accel, jerk_mag = sim.get_next_step(t, 10.0)
            if np.isnan(pos).any() or np.isnan(vel).any(): continue

            r_mag = np.linalg.norm(pos)
            v_mag = np.linalg.norm(vel)
            a_mag = np.linalg.norm(accel)
            z_ratio = vel[2] / (v_mag + 1e-8) if v_mag > 0 else 0
            speed_ratio = v_mag / max(1, r_mag * 0.0001)

            r_meas, _, _, m_bearing = generate_spherical_measurements(pos[0], pos[1], pos[2])
            if r_meas == -1: continue

            std_dev_x = max(5, 30.0 + (r_mag * np.radians(0.4)))
            std_dev_y = max(5, 30.0 + (r_mag * np.radians(0.4)))
            std_dev_z = max(10, 25.0 + (r_mag * np.radians(0.8)))

            svx = 0.3 + (a_mag * 0.05) if sim.track_class != "FIGHTER_JET" else 0.1
            svy = svx
            svz = 0.2
            rho = 0.6 if sim.track_class in ["MISSILE", "CIVILIAN"] else 0.3

            all_data.append({
                "track_no": sim.track_id, "track_class": sim.track_class, "manoeuvre_state": sim.state,
                "pos_x": pos[0], "pos_y": pos[1], "pos_z": pos[2],
                "vel_x": vel[0], "vel_y": vel[1], "vel_z": vel[2],
                "accel_x": accel[0], "accel_y": accel[1], "accel_z": accel[2],
                "accel_mag": a_mag, "jerk_mag": jerk_mag,
                "z_ratio": z_ratio, "speed_ratio": speed_ratio,
                "range_mag": r_mag, "vel_mag": v_mag,
                "std_dev_x": std_dev_x, "std_dev_y": std_dev_y, "std_dev_z": std_dev_z,
                "std_dev_vx": svx, "std_dev_vy": svy, "std_dev_vz": svz,
                "cov_x_vx": rho * std_dev_x * svx, "cov_y_vy": rho * std_dev_y * svy, "cov_z_vz": rho * std_dev_z * svz,
                "meas_range": r_meas, "meas_bearing": m_bearing,
                "ntp_sec": int(current_ntp_time), "ntp_frac": int((current_ntp_time % 1) * (2**32)),
                "time_step": t
            })

    df = pd.DataFrame(all_data)
    for track_id in df['track_no'].unique():
        track_mask = df['track_no'] == track_id
        if len(df[track_mask]) > 10:
            df.loc[track_mask, 'meas_range'] += np.random.normal(0, 20, sum(track_mask))
    return df

def build_improved_model(input_shape, num_classes):
    inputs = Input(shape=input_shape)
    x = Masking(mask_value=0.0)(inputs)
    x = Conv1D(32, kernel_size=3, activation='relu', padding='same')(x)
    x = BatchNormalization()(x)
    x = MaxPooling1D(pool_size=2)(x)
    lstm_out = Bidirectional(LSTM(64, return_sequences=True, dropout=0.2, recurrent_dropout=0.1))(x)
    attention_output = Attention()([lstm_out, lstm_out])
    attention_vector = GlobalAveragePooling1D()(attention_output)
    second_lstm = Bidirectional(LSTM(32, return_sequences=False, dropout=0.1))(lstm_out)
    combined = tf.keras.layers.concatenate([attention_vector, second_lstm])
    dense = Dense(64, activation='relu', kernel_regularizer=tf.keras.regularizers.l2(0.001))(combined)
    dense = Dropout(0.3)(dense)
    dense = BatchNormalization()(dense)
    outputs = Dense(num_classes, activation='softmax')(dense)

    model = Model(inputs=inputs, outputs=outputs)
    model.compile(optimizer=Adam(learning_rate=0.001), loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    return model

def add_additional_features(df):
    """Add enhanced features without manual scaling to prevent data leakage"""
    df = df.copy()
    df['curvature'] = np.sqrt((df['vel_x']**2 + df['vel_y']**2)) / (df['range_mag'] + 1e-8)
    df['maneuver_intensity'] = np.sqrt(df['accel_mag']**2 + df['jerk_mag']**2)
    df['heading'] = np.degrees(np.arctan2(df['vel_y'], df['vel_x'])) % 360

    def calculate_velocity_turn_rate(group):
        velocities = group[['vel_x', 'vel_y', 'vel_z']].values
        ntp_times = group['ntp_sec'].values
        if len(velocities) < 2: return np.zeros(len(velocities))

        turn_rates = []
        for i in range(1, len(velocities)):
            dt = ntp_times[i] - ntp_times[i-1]
            if dt > 0 and not np.isnan(velocities[i]).any() and not np.isnan(velocities[i-1]).any():
                v1, v2 = velocities[i-1], velocities[i]
                norm_v1, norm_v2 = np.linalg.norm(v1), np.linalg.norm(v2)
                if norm_v1 > 0 and norm_v2 > 0:
                    cos_angle = np.clip(np.dot(v1, v2) / (norm_v1 * norm_v2), -1, 1)
                    turn_rate = np.degrees(np.arccos(cos_angle)) / dt
                else: turn_rate = 0.0
            else: turn_rate = 0.0
            turn_rates.append(turn_rate)
        return np.array([0.0] + turn_rates)

    df['velocity_turn_rate'] = df.groupby('track_no').apply(calculate_velocity_turn_rate).reset_index(level=0, drop=True)

    rolling_window = 3
    df['accel_mag_rolling_mean'] = df.groupby('track_no')['accel_mag'].rolling(window=rolling_window).mean().reset_index(level=0, drop=True)
    df['accel_mag_rolling_std'] = df.groupby('track_no')['accel_mag'].rolling(window=rolling_window).std().reset_index(level=0, drop=True)

    def calculate_direction_changes(group):
        headings = group['heading'].values
        ntp_times = group['ntp_sec'].values
        if len(headings) < 2: return np.zeros(len(headings))

        direction_changes = []
        for i in range(1, len(headings)):
            dt = ntp_times[i] - ntp_times[i-1]
            if dt > 0 and not pd.isna(headings[i]) and not pd.isna(headings[i-1]):
                delta_heading = abs((headings[i] - headings[i-1]) % 360)
                direction_changes.append(min(delta_heading, 360-delta_heading) > 45)
            else: direction_changes.append(0)
        return np.array([0.0] + [float(x) for x in direction_changes])

    df['direction_change_rate'] = df.groupby('track_no').apply(calculate_direction_changes).reset_index(level=0, drop=True)
    df['vertical_accel_ratio'] = df['accel_z'] / (df['accel_mag'] + 1e-8)
    df['speed_change_rate'] = df.groupby('track_no')['vel_mag'].diff().fillna(0)
    df['position_ratio_xz'] = df['pos_x'] / (df['pos_z'] + 1e-8) if not df['pos_z'].eq(0).all() else 0.0

    # Fill Nans safely but do NOT manually normalize. The scaler handles this.
    for col in ['curvature', 'maneuver_intensity', 'velocity_turn_rate',
              'accel_mag_rolling_mean', 'accel_mag_rolling_std',
              'direction_change_rate', 'vertical_accel_ratio',
              'speed_change_rate', 'position_ratio_xz']:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0.0)

    return df

def prepare_sequences_and_split(df, seq_length=30, test_size=0.2):
    feature_cols = [
        'pos_z', 'vel_x', 'vel_y', 'vel_z',
        'accel_mag', 'jerk_mag', 'z_ratio',
        'range_mag', 'vel_mag', 'speed_ratio',
        'std_dev_x','std_dev_y','std_dev_z', 'std_dev_vx','std_dev_vy','std_dev_vz',
        'curvature', 'maneuver_intensity', 'velocity_turn_rate',
        'accel_mag_rolling_mean', 'accel_mag_rolling_std', 'direction_change_rate',
        'vertical_accel_ratio', 'speed_change_rate', 'position_ratio_xz'
    ]

    track_sequences, track_labels = [], []

    for track_id, group in df.groupby('track_no'):
        if len(group) >= 5:
            sorted_group = group.sort_values('ntp_sec')
            try:
                data_matrix = sorted_group[feature_cols].select_dtypes(include=[np.number]).values
                if np.isnan(data_matrix).any() or np.isinf(data_matrix).any():
                    data_matrix = np.nan_to_num(data_matrix, nan=0.0, posinf=0.0, neginf=0.0)

                max_start_idx = len(data_matrix) - seq_length + 1
                if max_start_idx > 0:
                    for i in range(max_start_idx):
                        sequence = data_matrix[i:i+seq_length]
                        track_sequences.append(sequence)
                        track_labels.append(sorted_group['track_class'].iloc[0])
                else:
                    # FIX 2: Pre-Pad sequences identically to test-time inference
                    if len(data_matrix) > 0:
                        padded_sequence = np.zeros((seq_length, data_matrix.shape[1]))
                        padded_sequence[-len(data_matrix):, :] = data_matrix
                        track_sequences.append(padded_sequence)
                        track_labels.append(sorted_group['track_class'].iloc[0])
            except Exception as e:
                continue

    return np.array(track_sequences), np.array(track_labels)

def train_improved_model(csv_path):
    print(f"Loading dataset from {csv_path}...")
    df = pd.read_csv(csv_path).dropna()

    print("Adding enhanced features...")
    df_enhanced = add_additional_features(df)
    X, y = prepare_sequences_and_split(df_enhanced, seq_length=30, test_size=0.2)

    le = LabelEncoder()
    y_encoded = le.fit_transform(y)

    if not os.path.exists("trained_model"): os.makedirs("trained_model")
    joblib.dump(le, "trained_model/label_encoder.pkl")

    X_train, X_test, y_train, y_test = train_test_split(X, y_encoded, test_size=0.2, random_state=42, stratify=y_encoded)

    # Proper Scaling Pipeline - Fixes Data Leakage
    X_train_reshaped = X_train.reshape(-1, X_train.shape[-1])
    scaler = StandardScaler()
    scaler.fit(X_train_reshaped)

    X_train_scaled = scaler.transform(X_train_reshaped).reshape(X_train.shape)
    X_test_scaled = scaler.transform(X_test.reshape(-1, X_test.shape[-1])).reshape(X_test.shape)

    joblib.dump(scaler, "trained_model/feature_scaler.pkl")
    print("Saved feature scaler to trained_model/feature_scaler.pkl")

    model = build_improved_model((X.shape[1], X.shape[2]), len(le.classes_))

    from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
    early_stopping = EarlyStopping(monitor='val_accuracy', patience=10, restore_best_weights=True, mode='max')
    reduce_lr = ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, min_lr=1e-7, verbose=1)
    checkpoint = ModelCheckpoint('trained_model/best_radar_model.keras', monitor='val_accuracy', save_best_only=True, mode='max', verbose=1)

    print("Training model...")
    history = model.fit(X_train_scaled, y_train, validation_data=(X_test_scaled, y_test),
                        epochs=10, batch_size=32, callbacks=[early_stopping, reduce_lr, checkpoint], verbose=1)

    model.save("trained_model/radar_target_classifier.keras")
    return model, history

if __name__ == "__main__":
    if not os.path.exists("trained_model"): os.makedirs("trained_model")
    enhanced_dataset = generate_enhanced_dataset(num_tracks=200, duration=150)
    enhanced_dataset.to_csv("enhanced_radar_dataset.csv", index=False)
    train_improved_model("enhanced_radar_dataset.csv")

Loading dataset from enhanced_radar_dataset.csv...
Adding enhanced features...
Saved feature scaler to trained_model/feature_scaler.pkl
Training model...
Epoch 1/10
2988/2989 ━━━━━━━━━━━━━━━━━━━━ 0s 59ms/step - accuracy: 0.9178 - loss: 0.3072
Epoch 1: val_accuracy improved from None to 0.98415, saving model to trained_model/best_radar_model.keras

Epoch 1: finished saving model to trained_model/best_radar_model.keras
2989/2989 ━━━━━━━━━━━━━━━━━━━━ 205s 64ms/step - accuracy: 0.9549 - loss: 0.1797 - val_accuracy: 0.9841 - val_loss: 0.0610 - learning_rate: 0.0010
Epoch 2/10
2988/2989 ━━━━━━━━━━━━━━━━━━━━ 0s 59ms/step - accuracy: 0.9778 - loss: 0.0797
Epoch 2: val_accuracy improved from 0.98415 to 0.98950, saving model to trained_model/best_radar_model.keras

Epoch 2: finished saving model to trained_model/best_radar_model.keras
2989/2989 ━━━━━━━━━━━━━━━━━━━━ 197s 62ms/step - accuracy: 0.9806 - loss: 0.0711 - val_accuracy: 0.9895 - val_loss: 0.0413 - learning_rate: 0.0010
Epoch 3/10
2988/2

2

In [2]:
import numpy as np
import pandas as pd
import random
from datetime import datetime, timedelta

def generate_spherical_measurements(pos_x, pos_y, pos_z):
    r_true = np.sqrt(pos_x**2 + pos_y**2 + pos_z**2)
    if r_true == 0: return 0.0, 0.0, 0.0, 0.0
    theta_true_rad = np.arctan2(pos_y, pos_x)
    phi_true_rad = np.arcsin(pos_z / r_true)
    sigma_r = 25.0
    sigma_theta_deg, sigma_phi_deg = 0.6, 1.0

    r_meas = max(0, r_true + np.random.normal(0, sigma_r))
    theta_meas_rad = theta_true_rad + np.random.normal(0, np.radians(sigma_theta_deg))
    phi_meas_rad = phi_true_rad + np.random.normal(0, np.radians(sigma_phi_deg))
    meas_bearing = (np.degrees(theta_meas_rad) + 360) % 360
    return r_meas, theta_meas_rad, phi_meas_rad, meas_bearing

def compute_std_devs(r_true):
    sigma_r = 20.0 + (r_true * 0.001)
    sigma_theta_deg = 0.5 + (r_true / 50000)
    sigma_phi_deg = 0.8 + (r_true / 100000)
    std_dev_x = max(5, sigma_r + (r_true * np.radians(sigma_theta_deg)))
    std_dev_z = max(10, sigma_r + (r_true * np.radians(sigma_phi_deg)))
    return {"std_dev_x": std_dev_x, "std_dev_y": std_dev_x, "std_dev_z": std_dev_z}

def generate_edge_case_maneuver(track_class, vel, t):
    vx, vy, vz = vel
    if track_class == "MISSILE" and random.random() < 0.1:
        return np.array([vx * 0.95, vy * 0.95, np.random.normal(-350, 60)])
    if track_class == "HELICOPTER" and random.random() < 0.12:
        if random.choice(["CLIMB", "TURN"]) == "CLIMB": return np.array([vx, vy, vz + np.random.normal(35, 8)])
        else:
            angle = random.uniform(0, 2 * np.pi)
            mag = np.linalg.norm([vx, vy])
            return np.array([mag * np.cos(angle) + np.random.normal(0, 10), mag * np.sin(angle) + np.random.normal(0, 10), vz])
    if track_class == "FIGHTER_JET" and random.random() < 0.2:
        acceleration_magnitude = np.random.uniform(90, 110)
        angle_change = random.uniform(-70, 70)
        c, s = np.cos(np.radians(angle_change)), np.sin(np.radians(angle_change))
        new_vx, new_vy = vx*c - vy*s, vx*s + vy*c
        speed_direction = np.arctan2(vel[1], vel[0])
        accel_x = np.cos(speed_direction) * acceleration_magnitude + np.random.normal(0, 5)
        accel_y = np.sin(speed_direction) * acceleration_magnitude + np.random.normal(0, 5)
        return np.array([new_vx + accel_x, new_vy + accel_y, vz])
    return np.array([vx, vy, vz])

def generate_civilian_trajectory(start_pos, start_vel, duration=50, dt=10.0):
    pos_x, pos_y, pos_z = start_pos
    vel_x, vel_y, vel_z = start_vel
    px, py, pz, vx, vy, vz = [], [], [], [], [], []
    for t in range(duration):
        if random.random() < 0.05:
            vel_x += np.random.normal(0, 5)
            vel_y += np.random.normal(0, 5)
            vel_z += np.random.normal(0, 1)
        speed = np.sqrt(vel_x**2 + vel_y**2 + vel_z**2)
        if speed > 650:
            scale = 650 / speed
            vel_x *= scale; vel_y *= scale; vel_z *= scale
        pos_x += vel_x * dt + np.random.normal(0, 1)
        pos_y += vel_y * dt + np.random.normal(0, 1)
        pos_z = max(5000, pos_z + vel_z * dt + np.random.normal(0, 1))
        px.append(pos_x); py.append(pos_y); pz.append(pos_z)
        vx.append(vel_x); vy.append(vel_y); vz.append(vel_z)
    return px, py, pz, vx, vy, vz

def generate_test_trajectory(track_class, duration=50, dt=10.0):
    # Fixed Missing/Null Lists crash
    pos_x_list, pos_y_list, pos_z_list = [], [], []
    vel_x_list, vel_y_list, vel_z_list = [], [], []
    accel_x_list, accel_y_list, accel_z_list = [], [], []
    range_mag_list, vel_mag_list, accel_mag_list, jerk_mag_list = [], [], [], []
    z_ratio_list, speed_ratio_list = [], []
    std_dev_x_list, std_dev_y_list, std_dev_z_list = [], [], []
    std_dev_vx_list, std_dev_vy_list, std_dev_vz_list = [], [], []
    cov_x_vx_list, cov_y_vy_list, cov_z_vz_list = [], [], []
    track_no_list, track_class_list, manoeuvre_state_list, ntp_sec_list = [], [], [], []

    track_id = random.randint(10000, 99999)
    pos_x, pos_y = np.random.uniform(-50000, 50000), np.random.uniform(-50000, 50000)

    if track_class == "SKIFF": pos_z = np.random.uniform(0, 2)
    elif track_class in ["FIGHTER_JET", "HELICOPTER"]: pos_z = np.random.uniform(500, 15000)
    else: pos_z = np.random.uniform(8000, 30000)

    if track_class == "FIGHTER_JET": v_mag = np.random.uniform(200, 700)
    elif track_class == "SKIFF": v_mag = np.random.uniform(5, 25)
    elif track_class == "MISSILE": v_mag = np.random.uniform(400, 1000)
    elif track_class == "HELICOPTER": v_mag = np.random.uniform(50, 100)
    else: v_mag = np.random.uniform(300, 500)

    angle = random.uniform(0, 2 * np.pi)
    vel = np.array([v_mag * np.cos(angle), v_mag * np.sin(angle), 0.0])

    # Pre-calculate base trajectory types to avoid list appending errors
    if track_class == "CIVILIAN":
        civ_px, civ_py, civ_pz, civ_vx, civ_vy, civ_vz = generate_civilian_trajectory(
            [pos_x, pos_y, pos_z], [vel[0], vel[1], vel[2]], duration=duration, dt=dt
        )
    else:
        accel = np.array([np.random.uniform(-3, 3), np.random.uniform(-3, 3), np.random.uniform(-1, 1)])

    start_time = 3991182426

    for t in range(duration):
        if track_class == "CIVILIAN":
            pos_x, pos_y, pos_z = civ_px[t], civ_py[t], civ_pz[t]
            vel = np.array([civ_vx[t], civ_vy[t], civ_vz[t]])
            accel = np.zeros(3) # Civilian assumption
        else:
            old_vel = vel.copy()
            vel = generate_edge_case_maneuver(track_class, vel, t)
            pos_x += vel[0] * dt + 0.5 * accel[0] * (dt**2) + np.random.normal(0, 2)
            pos_y += vel[1] * dt + 0.5 * accel[1] * (dt**2) + np.random.normal(0, 2)

            if track_class == "SKIFF":
                pos_z = max(0, np.sin(t * 0.3) * random.uniform(2, 8))
                vel[2] = 0.0
            else:
                pos_z += vel[2] * dt + 0.5 * accel[2] * (dt**2)

            pos_x += np.random.normal(0, 1.5)
            pos_y += np.random.normal(0, 1.5)
            pos_z += np.random.normal(0, 1.0)

        # Fixed Acceleration & Jerk Magnitude Errors
        r_mag = np.sqrt(pos_x**2 + pos_y**2 + pos_z**2)
        v_mag_curr = np.linalg.norm(vel)
        a_mag = np.linalg.norm(accel)

        if t > 0 and track_class != "CIVILIAN":
            jerk_mag = np.linalg.norm((vel - old_vel) / dt)
        else: jerk_mag = 0.0

        std_devs = compute_std_devs(r_mag)
        svx = 0.5 + (0.15 * a_mag) if track_class != "FIGHTER_JET" else 0.2
        svy, svz = svx, 0.3
        rho = 0.4 + np.random.uniform(-0.1, 0.1)
        m_state = "CRUISING"
        timestamp = start_time + int(t * 10.0) + np.random.randint(-2, 3)

        pos_x_list.append(pos_x); pos_y_list.append(pos_y); pos_z_list.append(pos_z)
        vel_x_list.append(vel[0]); vel_y_list.append(vel[1]); vel_z_list.append(vel[2])
        accel_x_list.append(accel[0]); accel_y_list.append(accel[1]); accel_z_list.append(accel[2])
        range_mag_list.append(r_mag); vel_mag_list.append(v_mag_curr)
        accel_mag_list.append(a_mag); jerk_mag_list.append(jerk_mag)

        z_ratio = vel[2] / (v_mag_curr + 1e-8) if v_mag_curr > 0 else 0
        speed_ratio = v_mag_curr / max(1, r_mag * 0.0001)
        z_ratio_list.append(z_ratio); speed_ratio_list.append(speed_ratio)

        std_dev_x_list.append(std_devs["std_dev_x"]); std_dev_y_list.append(std_devs["std_dev_y"]); std_dev_z_list.append(std_devs["std_dev_z"])
        std_dev_vx_list.append(svx); std_dev_vy_list.append(svy); std_dev_vz_list.append(svz)
        cov_x_vx_list.append(rho * std_devs["std_dev_x"] * svx)
        cov_y_vy_list.append(rho * std_devs["std_dev_y"] * svy)
        cov_z_vz_list.append(rho * std_devs["std_dev_z"] * svz)

        track_no_list.append(track_id); track_class_list.append(track_class)
        manoeuvre_state_list.append(m_state); ntp_sec_list.append(timestamp)

    df_data = {
        'pos_x': pos_x_list, 'pos_y': pos_y_list, 'pos_z': pos_z_list,
        'vel_x': vel_x_list, 'vel_y': vel_y_list, 'vel_z': vel_z_list,
        'accel_x': accel_x_list, 'accel_y': accel_y_list, 'accel_z': accel_z_list,
        'range_mag': range_mag_list, 'vel_mag': vel_mag_list, 'accel_mag': accel_mag_list, 'jerk_mag': jerk_mag_list,
        'z_ratio': z_ratio_list, 'speed_ratio': speed_ratio_list,
        'std_dev_x': std_dev_x_list, 'std_dev_y': std_dev_y_list, 'std_dev_z': std_dev_z_list,
        'std_dev_vx': std_dev_vx_list, 'std_dev_vy': std_dev_vy_list, 'std_dev_vz': std_dev_vz_list,
        'cov_x_vx': cov_x_vx_list, 'cov_y_vy': cov_y_vy_list, 'cov_z_vz': cov_z_vz_list,
        'track_no': track_no_list, 'track_class': track_class_list,
        'manoeuvre_state': manoeuvre_state_list, 'ntp_sec': ntp_sec_list
    }

    df_data['packet_size'] = [166] * len(df_data['pos_x'])
    df_data['source'] = [7] * len(df_data['pos_x'])
    df_data['destination'] = [255] * len(df_data['pos_x'])
    df_data['packet_type'] = [163] * len(df_data['pos_x'])
    df_data['packet_category'] = [11] * len(df_data['pos_x'])
    df_data['radar_id'] = [1] * len(df_data['pos_x'])

    meas_range_list, meas_bearing_list = [], []
    for i in range(len(df_data['pos_x'])):
        if random.random() < 0.03:
            meas_range_list.append(-1); meas_bearing_list.append(-1)
        else:
            meas_range_list.append(df_data['range_mag'][i])
            meas_bearing_list.append(np.degrees(np.arctan2(vel_y_list[i], vel_x_list[i])) % 360)

    df_data['meas_range'] = meas_range_list
    df_data['meas_bearing'] = meas_bearing_list
    df_data['ntp_frac'] = [int((timestamp % 1) * (2**32)) for timestamp in ntp_sec_list]
    df_data['track_category_val'] = [1] * len(df_data['pos_x'])
    df_data['track_type'] = [5] * len(df_data['pos_x'])
    df_data['track_state'] = [0] * len(df_data['pos_x'])
    df_data['hostility'] = [0] * len(df_data['pos_x'])
    df_data['is_simulated'] = [1] * len(df_data['pos_x'])
    df_data['mod_cms_track'] = [0] * len(df_data['pos_x'])
    df_data['assoc_indication'] = [1] * len(df_data['pos_x'])

    df = pd.DataFrame(df_data)
    return df if len(df) > 0 else None

def generate_test_dataset(num_samples=100, duration=50):
    dfs = []
    track_classes = ["FIGHTER_JET", "SKIFF", "MISSILE", "HELICOPTER", "CIVILIAN"]
    samples_per_class = num_samples // len(track_classes)
    remaining_samples = num_samples % len(track_classes)

    for i, class_choice in enumerate(track_classes):
        actual_samples = samples_per_class + (1 if i < remaining_samples else 0)
        for j in range(actual_samples):
            df = generate_test_trajectory(class_choice, duration=duration)
            if df is not None and len(df) > 0: dfs.append(df)

    return pd.concat(dfs, ignore_index=True)

if __name__ == "__main__":
    test_df = generate_test_dataset(num_samples=100, duration=50)
    test_df.to_csv("test_radar_data.csv", index=False)
    print(f"Successfully generated 'test_radar_data.csv' with {len(test_df)} rows.")

Successfully generated 'test_radar_data.csv' with 5000 rows.


In [3]:
import pandas as pd
import numpy as np
from tensorflow.keras.models import load_model
import joblib

def prepare_features_for_inference(df):
    df = df.copy()
    if 'vel_z' in df.columns and 'vel_mag' in df.columns: df['z_ratio'] = df['vel_z'] / (df['vel_mag'] + 1e-8)
    else: df['z_ratio'] = 0.0

    if 'vel_mag' in df.columns and 'range_mag' in df.columns: df['speed_ratio'] = df['vel_mag'] / (df['range_mag'] + 1e-8)
    else: df['speed_ratio'] = 0.0

    if 'vel_x' in df.columns and 'vel_y' in df.columns and 'range_mag' in df.columns: df['curvature'] = np.sqrt((df['vel_x']**2 + df['vel_y']**2)) / (df['range_mag'] + 1e-8)
    else: df['curvature'] = 0.0

    if 'accel_mag' in df.columns and 'jerk_mag' in df.columns: df['maneuver_intensity'] = np.sqrt(df['accel_mag']**2 + df['jerk_mag']**2)
    else: df['maneuver_intensity'] = 0.0

    if 'vel_x' in df.columns and 'vel_y' in df.columns: df['heading'] = np.degrees(np.arctan2(df['vel_y'], df['vel_x'])) % 360
    else: df['heading'] = 0.0

    if 'vel_x' in df.columns and 'vel_y' in df.columns and 'ntp_sec' in df.columns:
        try:
            def calculate_velocity_turn_rate(group):
                velocities = group[['vel_x', 'vel_y', 'vel_z']].values
                ntp_times = group['ntp_sec'].values
                if len(velocities) < 2: return np.zeros(len(velocities))
                turn_rates = []
                for i in range(1, len(velocities)):
                    dt = ntp_times[i] - ntp_times[i-1]
                    if dt > 0 and not np.isnan(velocities[i]).any() and not np.isnan(velocities[i-1]).any():
                        v1, v2 = velocities[i-1], velocities[i]
                        norm_v1, norm_v2 = np.linalg.norm(v1), np.linalg.norm(v2)
                        if norm_v1 > 0 and norm_v2 > 0:
                            cos_angle = np.clip(np.dot(v1, v2) / (norm_v1 * norm_v2), -1, 1)
                            turn_rate = np.degrees(np.arccos(cos_angle)) / dt
                        else: turn_rate = 0.0
                    else: turn_rate = 0.0
                    turn_rates.append(turn_rate)
                return np.array(turn_rates + [0.0] * (len(velocities) - len(turn_rates)))
            df['velocity_turn_rate'] = df.groupby('track_no').apply(calculate_velocity_turn_rate).reset_index(level=0, drop=True)
        except: df['velocity_turn_rate'] = 0.0
    else: df['velocity_turn_rate'] = 0.0

    if 'accel_mag' in df.columns and 'track_no' in df.columns:
        try:
            df['accel_mag_rolling_mean'] = df.groupby('track_no')['accel_mag'].rolling(window=5).mean().reset_index(level=0, drop=True)
            df['accel_mag_rolling_std'] = df.groupby('track_no')['accel_mag'].rolling(window=5).std().reset_index(level=0, drop=True)
        except: df['accel_mag_rolling_mean'], df['accel_mag_rolling_std'] = 0.0, 0.0
    else: df['accel_mag_rolling_mean'], df['accel_mag_rolling_std'] = 0.0, 0.0

    if 'heading' in df.columns and 'ntp_sec' in df.columns and 'track_no' in df.columns:
        try:
            def calculate_direction_changes(group):
                headings, ntp_times = group['heading'].values, group['ntp_sec'].values
                if len(headings) < 2: return np.zeros(len(headings))
                direction_changes = []
                for i in range(1, len(headings)):
                    dt = ntp_times[i] - ntp_times[i-1]
                    if dt > 0 and not pd.isna(headings[i]) and not pd.isna(headings[i-1]):
                        delta_heading = abs((headings[i] - headings[i-1]) % 360)
                        direction_changes.append(min(delta_heading, 360-delta_heading) > 45)
                    else: direction_changes.append(0)
                return np.array(direction_changes + [0.0] * (len(headings) - len(direction_changes)))
            df['direction_change_rate'] = df.groupby('track_no').apply(calculate_direction_changes).reset_index(level=0, drop=True)
        except: df['direction_change_rate'] = 0.0
    else: df['direction_change_rate'] = 0.0

    if 'accel_z' in df.columns and 'accel_mag' in df.columns: df['vertical_accel_ratio'] = df['accel_z'] / (df['accel_mag'] + 1e-8)
    else: df['vertical_accel_ratio'] = 0.0

    if 'vel_mag' in df.columns and 'track_no' in df.columns:
        try: df['speed_change_rate'] = df.groupby('track_no')['vel_mag'].diff().fillna(0)
        except: df['speed_change_rate'] = 0.0
    else: df['speed_change_rate'] = 0.0

    if 'pos_x' in df.columns and 'pos_z' in df.columns:
        try: df['position_ratio_xz'] = df['pos_x'] / (df['pos_z'] + 1e-8)
        except: df['position_ratio_xz'] = 0.0
    else: df['position_ratio_xz'] = 0.0

    numeric_columns = ['heading', 'jerk_mag', 'velocity_turn_rate', 'accel_mag_rolling_mean', 'accel_mag_rolling_std', 'direction_change_rate', 'vertical_accel_ratio', 'speed_change_rate', 'position_ratio_xz', 'curvature', 'maneuver_intensity', 'z_ratio', 'speed_ratio']
    for col in numeric_columns:
        if col in df.columns: df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0.0)

    return df

def predict_single_track(track_group, model, le, scaler):
    feature_cols = [
        'pos_z', 'vel_x', 'vel_y', 'vel_z',
        'accel_mag', 'jerk_mag', 'z_ratio',
        'range_mag', 'vel_mag', 'speed_ratio',
        'std_dev_x','std_dev_y','std_dev_z', 'std_dev_vx','std_dev_vy','std_dev_vz',
        'curvature', 'maneuver_intensity', 'velocity_turn_rate',
        'accel_mag_rolling_mean', 'accel_mag_rolling_std', 'direction_change_rate',
        'vertical_accel_ratio', 'speed_change_rate', 'position_ratio_xz'
    ]

    available_cols = [col for col in feature_cols if col in track_group.columns]

    try:
        features = track_group[available_cols].values
        if len(features) == 0: return {'prediction': 'UNKNOWN', 'confidence': 0.0}

        # Uses standard scaler matching the training pipeline
        features_scaled = scaler.transform(features)

        seq_length = 30
        if len(features_scaled) >= seq_length:
            input_data = features_scaled[-seq_length:].reshape(1, seq_length, -1)
        else:
            # Padded at the beginning to align with training
            padded = np.zeros((1, seq_length, len(available_cols)))
            padded[0, -(len(features_scaled)):, :] = features_scaled
            input_data = padded

        pred_probs = model.predict(input_data, verbose=0)[0]
        predicted_class_idx = np.argmax(pred_probs)
        confidence = np.max(pred_probs)
        predicted_label = le.inverse_transform([predicted_class_idx])[0]

        return {'prediction': predicted_label, 'confidence': float(confidence)}
    except Exception as e:
        return {'prediction': f'ERROR: {str(e)}', 'confidence': 0.0}

def process_test_file(file_path):
    print("Loading trained model...")
    model = load_model("trained_model/radar_target_classifier.keras", compile=False)
    le = joblib.load("trained_model/label_encoder.pkl")
    scaler = joblib.load("trained_model/feature_scaler.pkl")
    print("✓ Model and encoders loaded successfully")

    df_test = pd.read_csv(file_path).dropna(subset=['track_no'])
    df_test = prepare_features_for_inference(df_test)
    final_results = []

    for track_id, group in df_test.groupby('track_no'):
        actual_label = group['track_class'].iloc[0] if 'track_class' in group.columns else "N/A"
        pred_res = predict_single_track(group, model, le, scaler)
        final_results.append({
            'Track ID': track_id, 'Actual Class': actual_label, 'Predicted Class': pred_res['prediction'],
            'Confidence': f"{pred_res['confidence']:.4f}", 'Result': '✅ Correct' if actual_label == pred_res['prediction'] else '❌ Wrong'
        })
    return final_results

def visualize_results_table(results):
    df_res = pd.DataFrame(results)
    correct_count = len(df_res[df_res['Result'] == '✅ Correct'])
    total = len(df_res)
    accuracy = (correct_count / total) * 100 if total > 0 else 0

    print("\n" + "="*60)
    print(f" FINAL CLASSIFICATION REPORT ")
    print("="*60)
    print(f"Total Tracks: {total} | Correct: {correct_count} | Accuracy: {accuracy:.2f}%")
    print("-" * 60)
    print(df_res.to_string(index=False))
    print("="*60)
    df_res.to_csv("prediction_comparison_results.csv", index=False)
    print("\nDetailed results saved to 'prediction_comparison_results.csv'")

if __name__ == "__main__":
    results_list = process_test_file("test_radar_data.csv")
    if results_list: visualize_results_table(results_list)

Loading trained model...
✓ Model and encoders loaded successfully

 FINAL CLASSIFICATION REPORT 
Total Tracks: 100 | Correct: 60 | Accuracy: 60.00%
------------------------------------------------------------
 Track ID Actual Class Predicted Class Confidence    Result
    10102     CIVILIAN        CIVILIAN     0.7766 ✅ Correct
    10229      MISSILE        CIVILIAN     0.9999   ❌ Wrong
    10625   HELICOPTER      HELICOPTER     1.0000 ✅ Correct
    11489     CIVILIAN        CIVILIAN     0.9992 ✅ Correct
    15120        SKIFF      HELICOPTER     0.8755   ❌ Wrong
    15924     CIVILIAN        CIVILIAN     0.9999 ✅ Correct
    17052        SKIFF        CIVILIAN     0.9999   ❌ Wrong
    17339  FIGHTER_JET     FIGHTER_JET     0.9999 ✅ Correct
    18132  FIGHTER_JET     FIGHTER_JET     1.0000 ✅ Correct
    18356  FIGHTER_JET     FIGHTER_JET     1.0000 ✅ Correct
    19092      MISSILE        CIVILIAN     1.0000   ❌ Wrong
    24743     CIVILIAN         MISSILE     0.4687   ❌ Wrong
    26170  